# Customer Churn Analysis

Exploratory data analysis, segmentation, and storytelling.

Run `python ../src/generate_data.py` first (or drop the real Kaggle CSV into `../data/telco_churn.csv`).

## 1. Setup & load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

df = pd.read_csv('../data/telco_churn.csv')
print(df.shape)
df.head()

In [ ]:
df.info()
df.isna().sum()

## 2. Clean & engineer features

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop(columns=['customerID'])

for col in df.select_dtypes('object').columns:
    df[col] = df[col].replace({'No internet service': 'No', 'No phone service': 'No'})

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df['tenure_group'] = pd.cut(df['tenure'], bins=[-1,12,24,48,60,72],
                            labels=['0-12m','12-24m','24-48m','48-60m','60-72m'])
service_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['num_services'] = (df[service_cols] == 'Yes').sum(axis=1)
df['charges_per_tenure'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
df.head()

## 3. EDA

### Baseline churn

In [ ]:
baseline = df['Churn'].mean()
print(f'Overall churn rate: {baseline:.1%}')
sns.countplot(data=df, x='Churn')
plt.title('Churned (1) vs Retained (0)')
plt.show()

### Churn by category

In [ ]:
def churn_by(col):
    g = (df.groupby(col, observed=True)['Churn']
           .agg(churn_rate='mean', customers='size')
           .sort_values('churn_rate', ascending=False))
    g['vs_baseline'] = g['churn_rate'] - baseline
    return g.round(3)

churn_by('Contract')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flat, ['Contract','InternetService','PaymentMethod','tenure_group']):
    (df.groupby(col, observed=True)['Churn'].mean().sort_values()
        .plot(kind='barh', ax=ax, color='#d1495b'))
    ax.axvline(baseline, ls='--', color='gray', label='baseline')
    ax.set_title(f'Churn rate by {col}'); ax.legend()
plt.tight_layout(); plt.show()

### Numeric drivers & correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False, ax=axes[0])
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False, ax=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
sns.heatmap(df.select_dtypes('number').corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation'); plt.show()

## 4. Segmentation - the Wow factor

In [ ]:
segments = (df.groupby(['Contract','InternetService','TechSupport'], observed=True)['Churn']
              .agg(churn_rate='mean', customers='size').reset_index())
segments = segments[segments['customers'] >= 100].copy()
segments['risk_multiple'] = segments['churn_rate'] / baseline
segments.sort_values('churn_rate', ascending=False).head(10).round(3)

## 5. Story & revenue at stake

In [ ]:
seg = df[(df['Contract']=='Month-to-month') & (df['InternetService']=='Fiber optic') & (df['TechSupport']=='No')]
print(f"Segment churn: {seg['Churn'].mean():.1%}  ({seg['Churn'].mean()/baseline:.1f}x baseline)")
at_risk = seg['MonthlyCharges'].sum() * 12
saved = seg[seg['Churn']==1]['MonthlyCharges'].sum() * 12 * 0.25
print(f'Annual revenue exposed: ${at_risk:,.0f}')
print(f'Saved with 25% churn cut: ${saved:,.0f}')

### Retention strategy

1. Migrate month-to-month users to annual contracts with a discount.
2. 90-day onboarding journey for first-year customers.
3. Free tech-support trial for at-risk fiber users.
4. Incentivize auto-pay to move users off electronic check.
5. Proactive quality outreach for high-paying fiber customers.